# Statistical tests for Fig. 4 (Section 4.2)

- Wilcoxon signed-rank test (two-sided) for all pairs, with the p values adjusted by the Holm method in each comparison.
- The rows of all result files are in the same participant order.

In [1]:
import itertools as it

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

In [2]:
param_dir = "../param"
out_dir = f"{param_dir}/wilcoxon_holm"
env_lis = ["office", "supermarket", "office_supermarket"]
method_dict = {
    "I-VT (opt)": "ivtl_80deg_160ms_opt",
    "I-VT (vis)": "ivtl_80deg_160ms_vis",
    "I-DT (opt)": "idt_07deg_160ms_opt",
    "I-DT (vis)": "idt_07deg_160ms_vis",
    "I-VDT (opt)": "ivdt_80deg_07deg_160ms_opt",
    "I-VDT (vis)": "ivdt_80deg_07deg_160ms_vis",
}

In [3]:
def wilcoxon_holm(data):
    """Run the Wilcoxon signed-rank test for all pairs and adjust the p values with the Holm method."""
    rows = []
    for a, b in it.combinations(data, 2):
        r = stats.wilcoxon(data[a], data[b])
        rows.append({"group1": a, "group2": b, "W": r.statistic, "p_raw": r.pvalue})
    res = pd.DataFrame(rows)
    res["p_adj"] = multipletests(res["p_raw"], method="holm")[1]
    res["sig"] = np.select([res["p_adj"] < 0.01, res["p_adj"] < 0.05], ["**", "*"], "")
    return res

## Comparison of the six methods and the optical axis (for each environment)

In [4]:
opt_axis = pd.read_csv(f"{param_dir}/result_opt.csv")["error"].values
acc = {env: {name: pd.read_csv(f"{param_dir}/{folder}/result_{env}.csv")["error"].values for name, folder in method_dict.items()} for env in env_lis}

res = []
for env in env_lis:
    r = wilcoxon_holm({**acc[env], "Optical axis": opt_axis})
    r.insert(0, "env", env)
    res.append(r)
fig4_method = pd.concat(res, ignore_index=True)
fig4_method.to_csv(f"{out_dir}/fig4_method.csv", index=False)
fig4_method.round(4)

,env,group1,group2,W,p_raw,p_adj,sig
0,office,I-VT (opt),I-VT (vis),45.0,0.0814,0.5700,
1,office,I-VT (opt),I-DT (opt),78.0,0.7660,1.0000,
2,office,I-VT (opt),I-DT (vis),9.0,0.0003,0.0045,**
3,office,I-VT (opt),I-VDT (opt),72.0,0.5798,1.0000,
4,office,I-VT (opt),I-VDT (vis),15.0,0.0010,0.0167,*
...,...,...,...,...,...,...,...
58,office_supermarket,I-DT (vis),I-VDT (vis),30.0,0.2787,1.0000,
59,office_supermarket,I-DT (vis),Optical axis,0.0,0.0000,0.0002,**
60,office_supermarket,I-VDT (opt),I-VDT (vis),21.0,0.0034,0.0336,*
61,office_supermarket,I-VDT (opt),Optical axis,0.0,0.0000,0.0002,**


## Comparison of the environments (for each method)

In [5]:
res = []
for name in method_dict:
    r = wilcoxon_holm({env: acc[env][name] for env in env_lis})
    r.insert(0, "method", name)
    res.append(r)
fig4_env = pd.concat(res, ignore_index=True)
fig4_env.to_csv(f"{out_dir}/fig4_env.csv", index=False)
fig4_env.round(4)

,method,group1,group2,W,p_raw,p_adj,sig
0,I-VT (opt),office,supermarket,49.0,0.1187,0.2374,
1,I-VT (opt),office,office_supermarket,79.0,0.7987,0.7987,
2,I-VT (opt),supermarket,office_supermarket,40.0,0.0483,0.1448,
3,I-VT (vis),office,supermarket,42.0,0.0599,0.1295,
4,I-VT (vis),office,office_supermarket,84.0,0.9661,0.9661,
5,I-VT (vis),supermarket,office_supermarket,39.0,0.0432,0.1295,
6,I-DT (opt),office,supermarket,83.0,0.9323,1.0000,
7,I-DT (opt),office,office_supermarket,72.0,0.5798,1.0000,
8,I-DT (opt),supermarket,office_supermarket,52.0,0.1540,0.4621,
9,I-DT (vis),office,supermarket,77.0,0.7337,0.7337,
